# LJ EOS local analysis

Локальный notebook для обработки CSV, полученных production-runner-ом (`lj_vdw_notebook.ipynb`).

Чтобы добавить новые seed, положите файлы `eos_seed_<seed>.csv`, `eos_blocks_seed_<seed>.csv` и `failures_seed_<seed>.csv` в `local_data/raw/`, затем перезапустите этот notebook.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / 'local_data' / 'raw'
PROCESSED_DIR = PROJECT_DIR / 'local_data' / 'processed'
FIGURES_DIR = PROJECT_DIR / 'local_data' / 'figures'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('FIGURES_DIR:', FIGURES_DIR)


In [ ]:

def read_many(patterns):
    if isinstance(patterns, str):
        patterns = [patterns]
    paths = []
    for pattern in patterns:
        paths.extend(sorted(RAW_DIR.glob(pattern)))
    paths = sorted(dict.fromkeys(paths))
    if not paths:
        return pd.DataFrame(), []
    frames = []
    for path in paths:
        df = pd.read_csv(path)
        df['source_file'] = path.name
        frames.append(df)
    return pd.concat(frames, ignore_index=True), paths

eos_df, eos_files = read_many(['eos_seed_*.csv', 'eos_targeted*.csv'])
failures_df, failure_files = read_many(['failures_seed_*.csv', 'failures_targeted*.csv'])

targeted_mask = eos_df['source_file'].astype(str).str.startswith('eos_targeted') if len(eos_df) else pd.Series(dtype=bool)
seed_mask = eos_df['source_file'].astype(str).str.startswith('eos_seed') if len(eos_df) else pd.Series(dtype=bool)

print('EOS files:', [p.name for p in eos_files])
print('Failure files:', [p.name for p in failure_files])
print('EOS rows:', len(eos_df))
print('  seed-series EOS rows:', int(seed_mask.sum()))
print('  targeted EOS rows:', int(targeted_mask.sum()))
print('Failure rows:', len(failures_df))

if len(eos_df):
    eos_source_counts = eos_df.groupby('source_file').size().reset_index(name='rows').sort_values('source_file')
    eos_source_counts.to_csv(PROCESSED_DIR / 'eos_source_counts.csv', index=False)
    print('EOS rows by source file:')
    display(eos_source_counts)

if len(eos_df) == 0:
    raise RuntimeError('No EOS CSV files found in local_data/raw/. Expected eos_seed_*.csv or eos_targeted*.csv')
if int(targeted_mask.sum()) == 0:
    print('NOTE: no eos_targeted*.csv rows found. Put eos_targeted_points.csv into local_data/raw/ and rerun this notebook to include targeted runs.')


In [ ]:

# Базовая очистка и сводки. Не предполагаем, что сетка уже полная.
eos_ok = eos_df[eos_df.get('status', 'ok') == 'ok'].copy()
for col in ['seed', 'T_target', 'rho_target', 'P_mean', 'P_std', 'T_mean', 'U_mean', 'K_mean', 'E_mean', 'wall_time_s']:
    if col in eos_ok.columns:
        eos_ok[col] = pd.to_numeric(eos_ok[col], errors='coerce')

# Если в raw появятся пересекающиеся файлы, сначала схлопываем повторы внутри одного seed,
# чтобы один seed не получил больший вес в среднем по изотерме.
eos_points = (
    eos_ok.groupby(['seed', 'T_target', 'rho_target'], as_index=False)
    .agg(
        run_id=('run_id', 'first'),
        P_mean=('P_mean', 'mean'),
        P_std=('P_std', 'mean'),
        T_mean=('T_mean', 'mean'),
        U_mean=('U_mean', 'mean'),
        K_mean=('K_mean', 'mean'),
        E_mean=('E_mean', 'mean'),
        wall_time_s=('wall_time_s', 'mean'),
        n_source_rows=('run_id', 'size'),
    )
)

coverage = (
    eos_points.groupby(['seed', 'T_target'])['rho_target']
    .agg(n_densities='nunique', rho_min='min', rho_max='max')
    .reset_index()
    .sort_values(['seed', 'T_target'])
)

seed_summary = (
    eos_points.groupby('seed')
    .agg(n_points=('run_id', 'nunique'), n_temperatures=('T_target', 'nunique'), n_densities=('rho_target', 'nunique'), wall_time_s=('wall_time_s', 'sum'))
    .reset_index()
    .sort_values('seed')
)

def std_across(values):
    values = pd.Series(values).dropna()
    return float(values.std(ddof=1)) if len(values) > 1 else np.nan

eos_mean = (
    eos_points.groupby(['T_target', 'rho_target'])
    .agg(
        n_seeds=('seed', 'nunique'),
        P_mean=('P_mean', 'mean'),
        P_std_across_seeds=('P_mean', std_across),
        P_std_within_mean=('P_std', 'mean'),
        T_mean=('T_mean', 'mean'),
        U_mean=('U_mean', 'mean'),
        K_mean=('K_mean', 'mean'),
        E_mean=('E_mean', 'mean'),
    )
    .reset_index()
    .sort_values(['T_target', 'rho_target'])
)
eos_mean['P_err'] = eos_mean['P_std_across_seeds'].where(eos_mean['n_seeds'] > 1, eos_mean['P_std_within_mean'])
eos_mean['specific_volume'] = 1.0 / eos_mean['rho_target']

eos_ok.to_csv(PROCESSED_DIR / 'eos_all.csv', index=False)
eos_points.to_csv(PROCESSED_DIR / 'eos_points_by_seed_T_rho.csv', index=False)
coverage.to_csv(PROCESSED_DIR / 'coverage_by_seed_temperature.csv', index=False)
seed_summary.to_csv(PROCESSED_DIR / 'seed_summary.csv', index=False)
eos_mean.to_csv(PROCESSED_DIR / 'eos_mean_by_T_rho.csv', index=False)

if len(failures_df):
    failures_df.to_csv(PROCESSED_DIR / 'failures_all.csv', index=False)

print('seed_summary:')
display(seed_summary)
print('coverage:')
display(coverage.head(30))
print('mean EOS preview:')
display(eos_mean.head())


In [ ]:
# Экспорт одной изотермы для ноутбука рендеринга ролика.
# Этот CSV удобно загрузить в Kaggle как отдельный input dataset.
MOVIE_EOS_T_TARGET = 0.80
MOVIE_EOS_T_LABEL = f"T{int(round(MOVIE_EOS_T_TARGET * 100)):03d}"
MOVIE_EOS_CSV = PROCESSED_DIR / f"eos_for_compression_movie_{MOVIE_EOS_T_LABEL}.csv"

movie_eos = eos_points[np.isclose(eos_points['T_target'], MOVIE_EOS_T_TARGET, atol=1e-8)].copy()
movie_eos = movie_eos.sort_values(['rho_target', 'seed'])

if len(movie_eos) == 0:
    raise RuntimeError(f"No EOS rows found for T_target = {MOVIE_EOS_T_TARGET}")

movie_eos.to_csv(MOVIE_EOS_CSV, index=False)
print('Movie EOS temperature:', MOVIE_EOS_T_TARGET)
print('Movie EOS rows:', len(movie_eos))
print('Movie EOS densities:', sorted(movie_eos['rho_target'].unique()))
print('Saved Kaggle input CSV:', MOVIE_EOS_CSV)
display(movie_eos.head())


In [ ]:
# Визуализация EOS: давление по плотности для каждой температуры.
# Кресты погрешностей: по нескольким seed используется std между seed; для одного seed — P_std из production CSV.
plt.figure(figsize=(8, 5))
for T, part in eos_mean.groupby('T_target'):
    part = part.sort_values('rho_target')
    plt.errorbar(part['rho_target'], part['P_mean'], yerr=part['P_err'], marker='o', linewidth=1.4, capsize=3, label='T=' + format(float(T), '.2f'))
plt.xlabel('rho')
plt.ylabel('P')
plt.title('LJ EOS pressure isotherms')
plt.grid(alpha=0.3)
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
path = FIGURES_DIR / 'pressure_isotherms.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)


In [ ]:

# График зависимости давления от удельного объёма v = V/N = 1/rho.
plt.figure(figsize=(8, 5))
pv_df = eos_mean[(eos_mean['P_mean'] < 1.0) & (eos_mean['specific_volume'] < 60.0)].copy()
for T, part in pv_df.groupby('T_target'):
    part = part.sort_values('specific_volume')
    plt.errorbar(part['specific_volume'], part['P_mean'], yerr=part['P_err'], marker='o', linewidth=1.4, capsize=3, label='T=' + format(float(T), '.2f'))
plt.xlabel('v = V/N = 1/rho')
plt.ylabel('P')
plt.xlim(left=0, right=60)
plt.ylim(bottom=min(-0.5, float(pv_df['P_mean'].min()) * 1.05), top=1.0)
plt.title('Давление от удельного объёма, P < 1 и v < 60')
plt.grid(alpha=0.3)
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
path = FIGURES_DIR / 'pressure_specific_volume_isotherms.png'
plt.savefig(path, dpi=160)
print('shown points:', len(pv_df))
plt.show()
print('saved:', path)


## Приближённая спинодаль по немонотонным участкам

Ниже строится **приближённая спинодаль, восстановленная по границам немонотонных участков численных изотерм**. Для каждой температуры используется средняя по seed зависимость `P(rho)`. Спинодальные точки берутся только из исходной сетки плотностей: газовая граница — точка, после которой начинается убывание давления, плотная граница — точка, после которой давление снова начинает возрастать. Это не бинодаль и не точная термодинамическая граница метастабильности.


In [ ]:
SPINODAL_SLOPE_SIGMA = 1.0
SPINODAL_SEED_AGREEMENT_MIN = 0.60


def seed_negative_fraction(T, rho_left, rho_right):
    left = eos_points[(eos_points['T_target'] == T) & (eos_points['rho_target'] == rho_left)][['seed', 'P_mean']].rename(columns={'P_mean': 'P_left'})
    right = eos_points[(eos_points['T_target'] == T) & (eos_points['rho_target'] == rho_right)][['seed', 'P_mean']].rename(columns={'P_mean': 'P_right'})
    merged = left.merge(right, on='seed', how='inner')
    if len(merged) == 0:
        return np.nan, 0
    return float((merged['P_right'] < merged['P_left']).mean()), int(len(merged))


def significant_edge_masks(P, P_err):
    dP = np.diff(P)
    err_left = P_err[:-1]
    err_right = P_err[1:]
    combined_err = np.sqrt(err_left * err_left + err_right * err_right)
    combined_err = np.where(np.isfinite(combined_err), combined_err, 0.0)
    neg_sig = dP < -SPINODAL_SLOPE_SIGMA * combined_err
    pos_sig = dP > SPINODAL_SLOPE_SIGMA * combined_err
    return dP, combined_err, neg_sig, pos_sig

spinodal_rows = []
spinodal_point_rows = []
spinodal_status_rows = []

for T, part in eos_mean.groupby('T_target'):
    part = part.sort_values('rho_target').reset_index(drop=True)
    rho = part['rho_target'].to_numpy(dtype=float)
    P = part['P_mean'].to_numpy(dtype=float)
    P_err = part['P_err'].to_numpy(dtype=float)

    status = 'not_found'
    notes = ''
    gas_idx = None
    dense_idx = None

    if len(part) < 3:
        notes = 'too few density points'
    else:
        dP, combined_err, neg_sig, pos_sig = significant_edge_masks(P, P_err)
        negative_edges = np.where(dP < 0.0)[0]
        significant_negative_edges = np.where(neg_sig)[0]

        if len(negative_edges) == 0:
            notes = 'no measured decreasing pressure interval'
        elif len(significant_negative_edges) == 0:
            status = 'unreliable'
            notes = 'decreasing intervals are not larger than pressure uncertainty'
        else:
            first = int(significant_negative_edges[0])
            last = int(significant_negative_edges[-1])
            significant_positive_inside = bool(pos_sig[first:last + 1].any())
            has_recovery_after = bool(pos_sig[last + 1:].any()) if last + 1 < len(pos_sig) else False

            agreement_values = []
            agreement_counts = []
            for edge in significant_negative_edges:
                if first <= int(edge) <= last:
                    frac, count = seed_negative_fraction(T, rho[int(edge)], rho[int(edge) + 1])
                    if count > 0 and np.isfinite(frac):
                        agreement_values.append(frac)
                        agreement_counts.append(count)
            min_agreement = min(agreement_values) if agreement_values else np.nan
            min_count = min(agreement_counts) if agreement_counts else 0

            if significant_positive_inside:
                status = 'unreliable'
                notes = 'multiple significant decreasing segments separated by significant growth'
            elif not has_recovery_after:
                status = 'unreliable'
                notes = 'dense-side return to increasing pressure is not observed in measured grid'
            elif min_count >= 2 and min_agreement < SPINODAL_SEED_AGREEMENT_MIN:
                status = 'unreliable'
                notes = 'negative slope is not reproduced by enough seeds'
            else:
                status = 'ok'
                gas_idx = first
                dense_idx = last + 1
                notes = 'accepted measured boundaries of one nonmonotonic interval'

            if gas_idx is None:
                gas_idx = first
                dense_idx = last + 1

            spinodal_status_rows.append({
                'T_target': float(T),
                'status': status,
                'rho_gas_boundary': float(rho[gas_idx]) if gas_idx is not None else np.nan,
                'rho_dense_boundary': float(rho[dense_idx]) if dense_idx is not None and dense_idx < len(rho) else np.nan,
                'P_gas_boundary': float(P[gas_idx]) if gas_idx is not None else np.nan,
                'P_dense_boundary': float(P[dense_idx]) if dense_idx is not None and dense_idx < len(P) else np.nan,
                'n_density_points': int(len(part)),
                'n_negative_edges': int(len(negative_edges)),
                'n_significant_negative_edges': int(len(significant_negative_edges)),
                'min_seed_negative_fraction': float(min_agreement) if np.isfinite(min_agreement) else np.nan,
                'min_seed_pair_count': int(min_count),
                'notes': notes,
            })
            continue

    spinodal_status_rows.append({
        'T_target': float(T),
        'status': status,
        'rho_gas_boundary': np.nan,
        'rho_dense_boundary': np.nan,
        'P_gas_boundary': np.nan,
        'P_dense_boundary': np.nan,
        'n_density_points': int(len(part)),
        'n_negative_edges': 0,
        'n_significant_negative_edges': 0,
        'min_seed_negative_fraction': np.nan,
        'min_seed_pair_count': 0,
        'notes': notes,
    })

spinodal_status = pd.DataFrame(spinodal_status_rows).sort_values('T_target')
spinodal_table = spinodal_status[spinodal_status['status'] == 'ok'].copy()
spinodal_table = spinodal_table[[
    'T_target', 'rho_gas_boundary', 'rho_dense_boundary',
    'P_gas_boundary', 'P_dense_boundary', 'n_density_points', 'notes'
]]

for _, row in spinodal_table.iterrows():
    spinodal_point_rows.append({'branch': 'gas', 'T_target': row['T_target'], 'rho_target': row['rho_gas_boundary'], 'P_boundary': row['P_gas_boundary']})
    spinodal_point_rows.append({'branch': 'dense', 'T_target': row['T_target'], 'rho_target': row['rho_dense_boundary'], 'P_boundary': row['P_dense_boundary']})
spinodal_points = pd.DataFrame(spinodal_point_rows)

spinodal_table.to_csv(PROCESSED_DIR / 'approx_spinodal_nonmonotonic_boundaries.csv', index=False)
spinodal_status.to_csv(PROCESSED_DIR / 'approx_spinodal_temperature_status.csv', index=False)
spinodal_points.to_csv(PROCESSED_DIR / 'approx_spinodal_points_for_plot.csv', index=False)

print('Приближённая спинодаль, восстановленная по границам немонотонных участков численных изотерм:')
display(spinodal_table)
print('Температуры без автоматически принятой спинодали:')
display(spinodal_status[spinodal_status['status'] != 'ok'][['T_target', 'status', 'notes']])


In [ ]:
plt.figure(figsize=(7, 5))
if len(spinodal_table):
    gas = spinodal_table.sort_values('rho_gas_boundary')
    dense = spinodal_table.sort_values('rho_dense_boundary')
    plt.plot(gas['rho_gas_boundary'], gas['T_target'], marker='o', linewidth=1.6, label='gas-side branch')
    plt.plot(dense['rho_dense_boundary'], dense['T_target'], marker='s', linewidth=1.6, label='dense-side branch')
    plt.scatter(gas['rho_gas_boundary'], gas['T_target'], s=80, facecolors='none', edgecolors='black', label='measured boundary points')
    plt.scatter(dense['rho_dense_boundary'], dense['T_target'], s=80, facecolors='none', edgecolors='black')
else:
    plt.text(0.5, 0.5, 'No reliable nonmonotonic intervals detected', ha='center', va='center', transform=plt.gca().transAxes)

plt.xlabel('rho')
plt.ylabel('T')
plt.title('Приближённая спинодаль по границам немонотонных участков численных изотерм')
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
path = FIGURES_DIR / 'approx_spinodal_nonmonotonic_isotherms.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)


## Подгонка Ван-дер-Ваальса в тёплой газовой области

Fit выполняется только по точкам с T >= 1.1 и rho <= 0.2. Это не финальная физическая интерпретация, а локальная параметризация тёплой газовой области по уже собранным EOS-данным.

In [ ]:
from scipy.optimize import curve_fit
import json

VDW_T_MIN = 1.10
VDW_RHO_MAX = 0.20

vdw_fit_df = eos_mean[(eos_mean['T_target'] >= VDW_T_MIN) & (eos_mean['rho_target'] <= VDW_RHO_MAX)].copy()
vdw_fit_df = vdw_fit_df[np.isfinite(vdw_fit_df['P_mean']) & np.isfinite(vdw_fit_df['T_target']) & np.isfinite(vdw_fit_df['rho_target'])]

def vdw_pressure(x, a, b):
    T = x[:, 0]
    rho = x[:, 1]
    return rho * T / (1.0 - b * rho) - a * rho * rho

if len(vdw_fit_df) < 3:
    raise RuntimeError('Not enough warm gas points for Van der Waals fit')

xdata = vdw_fit_df[['T_target', 'rho_target']].to_numpy(dtype=float)
ydata = vdw_fit_df['P_mean'].to_numpy(dtype=float)

sigma = None
if 'P_err' in vdw_fit_df.columns:
    err = vdw_fit_df['P_err'].to_numpy(dtype=float)
    if np.all(np.isfinite(err)) and np.any(err > 0):
        sigma = np.where(err > 0, err, np.nanmedian(err[err > 0]))

popt, pcov = curve_fit(
    vdw_pressure,
    xdata,
    ydata,
    p0=[5.0, 1.0],
    bounds=([0.0, 0.0], [100.0, 1.0 / float(vdw_fit_df['rho_target'].max()) * 0.95]),
    sigma=sigma,
    absolute_sigma=False if sigma is not None else False,
    maxfev=20000,
)

a_fit, b_fit = [float(x) for x in popt]
vdw_fit_df['P_vdw_fit'] = vdw_pressure(xdata, a_fit, b_fit)
vdw_fit_df['P_residual'] = vdw_fit_df['P_mean'] - vdw_fit_df['P_vdw_fit']

rmse = float(np.sqrt(np.mean(vdw_fit_df['P_residual'] ** 2)))
mae = float(np.mean(np.abs(vdw_fit_df['P_residual'])))

fit_result = {
    'model': 'P = rho*T/(1 - b*rho) - a*rho^2',
    'region': {'T_min': VDW_T_MIN, 'rho_max': VDW_RHO_MAX},
    'n_points': int(len(vdw_fit_df)),
    'a': a_fit,
    'b': b_fit,
    'rmse_P': rmse,
    'mae_P': mae,
}

with open(PROCESSED_DIR / 'vdw_fit_warm_gas.json', 'w', encoding='utf-8') as f:
    json.dump(fit_result, f, ensure_ascii=False, indent=2)

vdw_fit_df.to_csv(PROCESSED_DIR / 'vdw_fit_warm_gas_points.csv', index=False)

print('Van der Waals warm gas fit:')
print(json.dumps(fit_result, ensure_ascii=False, indent=2))
display(vdw_fit_df[['T_target', 'rho_target', 'P_mean', 'P_vdw_fit', 'P_residual']].head())


## Устойчивость коэффициентов Ван-дер-Ваальса к области fit

Ниже основной fit не изменяется. Таблица нужна только для диагностики: как меняются коэффициенты при разных нижних границах температуры и верхних границах давления в низкоплотной области.


In [ ]:
# Диагностика устойчивости fit по сетке T_min и P_max.
# Основной результат выше остаётся fit_result для VDW_T_MIN и VDW_RHO_MAX.

VDW_STABILITY_T_MIN_VALUES = [1.10, 1.20, 1.30]
VDW_STABILITY_P_MAX_VALUES = [0.01, 0.02, 0.03]


def fit_vdw_for_pressure_region(T_min, P_max):
    region = eos_mean[
        (eos_mean['T_target'] >= float(T_min))
        & (eos_mean['P_mean'] > 0.0)
        & (eos_mean['P_mean'] <= float(P_max))
    ].copy()
    region = region[
        np.isfinite(region['P_mean'])
        & np.isfinite(region['T_target'])
        & np.isfinite(region['rho_target'])
    ]

    result = {
        'T_min': float(T_min),
        'P_max': float(P_max),
        'n_points': int(len(region)),
        'a': np.nan,
        'b': np.nan,
        'rmse_P': np.nan,
        'mae_P': np.nan,
        'status': 'ok',
    }

    if len(region) < 3:
        result['status'] = 'too_few_points'
        return result

    x = region[['T_target', 'rho_target']].to_numpy(dtype=float)
    y = region['P_mean'].to_numpy(dtype=float)

    sigma_local = None
    if 'P_err' in region.columns:
        err = region['P_err'].to_numpy(dtype=float)
        positive_err = err[np.isfinite(err) & (err > 0)]
        if len(positive_err):
            sigma_local = np.where(err > 0, err, np.nanmedian(positive_err))

    try:
        popt_local, _ = curve_fit(
            vdw_pressure,
            x,
            y,
            p0=[a_fit, b_fit],
            bounds=([0.0, 0.0], [100.0, 1.0 / float(region['rho_target'].max()) * 0.95]),
            sigma=sigma_local,
            absolute_sigma=False if sigma_local is not None else False,
            maxfev=20000,
        )
    except Exception as exc:
        result['status'] = 'fit_failed: ' + type(exc).__name__
        return result

    a_local, b_local = [float(value) for value in popt_local]
    residual = y - vdw_pressure(x, a_local, b_local)
    result.update({
        'a': a_local,
        'b': b_local,
        'rmse_P': float(np.sqrt(np.mean(residual ** 2))),
        'mae_P': float(np.mean(np.abs(residual))),
    })
    return result


vdw_stability_rows = []
for T_min in VDW_STABILITY_T_MIN_VALUES:
    for P_max in VDW_STABILITY_P_MAX_VALUES:
        vdw_stability_rows.append(fit_vdw_for_pressure_region(T_min, P_max))

vdw_stability_table = pd.DataFrame(vdw_stability_rows)
vdw_stability_table.to_csv(PROCESSED_DIR / 'vdw_fit_pressure_region_stability.csv', index=False)

display(vdw_stability_table)
print('saved:', PROCESSED_DIR / 'vdw_fit_pressure_region_stability.csv')


In [ ]:
# Диагностический график fit Ван-дер-Ваальса в области T >= 1.1, rho <= 0.2.
plt.figure(figsize=(8, 5))
for T, part in vdw_fit_df.groupby('T_target'):
    part = part.sort_values('rho_target')
    plt.errorbar(part['rho_target'], part['P_mean'], yerr=part['P_err'], fmt='o', capsize=3, label='data T=' + format(float(T), '.2f'))
    plt.plot(part['rho_target'], part['P_vdw_fit'], linewidth=1.6, label='vdW fit T=' + format(float(T), '.2f'))

plt.xlabel('rho')
plt.ylabel('P')
plt.title('Van der Waals fit: warm gas region')
plt.grid(alpha=0.3)
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
path = FIGURES_DIR / 'vdw_fit_warm_gas.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)

plt.figure(figsize=(7, 4))
plt.axhline(0.0, color='black', linewidth=1)
for T, part in vdw_fit_df.groupby('T_target'):
    part = part.sort_values('rho_target')
    plt.plot(part['rho_target'], part['P_residual'], marker='o', linewidth=1.2, label='T=' + format(float(T), '.2f'))
plt.xlabel('rho')
plt.ylabel('P_mean - P_vdw_fit')
plt.title('Van der Waals fit residuals')
plt.grid(alpha=0.3)
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
path = FIGURES_DIR / 'vdw_fit_warm_gas_residuals.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)


## Спинодаль модели Ван-дер-Ваальса поверх численных границ

Для fit `P = rho*T/(1 - b*rho) - a*rho^2` условие спинодали `(dP/drho)_T = 0` даёт `T = 2*a*rho*(1 - b*rho)^2`. Ниже эта модельная кривая строится в тех же осях, что и измеренные границы приближённой спинодали.


In [ ]:

# Диагностическое сравнение: спинодаль fit Ван-дер-Ваальса, бинодаль fit Ван-дер-Ваальса и измеренные границы.
# Это модельные кривые по коэффициентам fit, а не замена измеренной спинодали.

from scipy.optimize import brentq

rho_limit = 1.0 / b_fit
rho_grid = np.linspace(1e-5, rho_limit * 0.999, 1200)
T_vdw_spinodal = 2.0 * a_fit * rho_grid * (1.0 - b_fit * rho_grid) ** 2
critical_rho = 1.0 / (3.0 * b_fit)
critical_T = 8.0 * a_fit / (27.0 * b_fit)

vdw_spinodal_df = pd.DataFrame({
    'rho': rho_grid,
    'T_vdw_spinodal': T_vdw_spinodal,
})
vdw_spinodal_df.to_csv(PROCESSED_DIR / 'vdw_spinodal_curve.csv', index=False)

def vdw_pressure_rho(rho, T):
    return rho * T / (1.0 - b_fit * rho) - a_fit * rho * rho

def vdw_spinodal_equation(rho, T):
    return 2.0 * a_fit * rho * (1.0 - b_fit * rho) ** 2 - T

def vdw_maxwell_primitive(v, T, P0):
    return T * np.log(v - b_fit) + a_fit / v - P0 * v

def vdw_binodal_at_T(T):
    rho_low_spinodal = brentq(lambda r: vdw_spinodal_equation(r, T), 1e-12, critical_rho - 1e-12)
    rho_high_spinodal = brentq(lambda r: vdw_spinodal_equation(r, T), critical_rho + 1e-12, rho_limit - 1e-12)
    P_min = vdw_pressure_rho(rho_high_spinodal, T)
    P_max = vdw_pressure_rho(rho_low_spinodal, T)

    def roots_for_pressure(P0):
        # On the gas side the coexistence pressure may be very small, so bracket by scanning.
        scan = np.linspace(1e-10, rho_low_spinodal - 1e-10, 500)
        values = vdw_pressure_rho(scan, T) - P0
        crossings = np.where(values[:-1] * values[1:] <= 0.0)[0]
        if len(crossings) == 0:
            raise ValueError('no gas-side root')
        i = int(crossings[0])
        rho_gas = brentq(lambda r: vdw_pressure_rho(r, T) - P0, scan[i], scan[i + 1])
        rho_liquid = brentq(lambda r: vdw_pressure_rho(r, T) - P0, rho_high_spinodal + 1e-10, rho_limit - 1e-10)
        return rho_gas, rho_liquid

    def maxwell_area(P0):
        rho_gas, rho_liquid = roots_for_pressure(P0)
        v_gas = 1.0 / rho_gas
        v_liquid = 1.0 / rho_liquid
        return vdw_maxwell_primitive(v_gas, T, P0) - vdw_maxwell_primitive(v_liquid, T, P0)

    p_grid = np.linspace(max(1e-12, P_min + 1e-10), P_max - 1e-10, 240)
    last_P = None
    last_area = None
    for P0 in p_grid:
        try:
            area = maxwell_area(P0)
        except ValueError:
            continue
        if last_area is not None and last_area * area <= 0.0:
            P_coex = brentq(maxwell_area, last_P, P0)
            rho_gas, rho_liquid = roots_for_pressure(P_coex)
            return rho_gas, rho_liquid, P_coex
        last_P = P0
        last_area = area
    raise ValueError('no Maxwell root')

binodal_rows = []
for T in np.linspace(0.995 * critical_T, max(0.55, 0.45 * critical_T), 140):
    try:
        rho_gas, rho_liquid, P_coex = vdw_binodal_at_T(float(T))
        binodal_rows.append({
            'T': float(T),
            'rho_gas': float(rho_gas),
            'rho_liquid': float(rho_liquid),
            'P_coex': float(P_coex),
        })
    except ValueError:
        pass
vdw_binodal_df = pd.DataFrame(binodal_rows)
vdw_binodal_df.to_csv(PROCESSED_DIR / 'vdw_binodal_curve.csv', index=False)

plt.figure(figsize=(7, 5))
plt.plot(rho_grid, T_vdw_spinodal, color='black', linewidth=1.8, label='vdW spinodal from fitted a,b')
if len(vdw_binodal_df):
    plt.plot(vdw_binodal_df['rho_gas'], vdw_binodal_df['T'], color='black', linestyle='--', linewidth=1.4, label='vdW binodal from fitted a,b')
    plt.plot(vdw_binodal_df['rho_liquid'], vdw_binodal_df['T'], color='black', linestyle='--', linewidth=1.4)
plt.scatter([critical_rho], [critical_T], color='black', marker='x', s=70, label='vdW critical point')

if len(spinodal_table):
    gas = spinodal_table.sort_values('rho_gas_boundary')
    dense = spinodal_table.sort_values('rho_dense_boundary')
    plt.plot(gas['rho_gas_boundary'], gas['T_target'], marker='o', linewidth=1.4, label='measured gas-side boundary')
    plt.plot(dense['rho_dense_boundary'], dense['T_target'], marker='s', linewidth=1.4, label='measured dense-side boundary')
    plt.scatter(gas['rho_gas_boundary'], gas['T_target'], s=85, facecolors='none', edgecolors='tab:blue')
    plt.scatter(dense['rho_dense_boundary'], dense['T_target'], s=85, facecolors='none', edgecolors='tab:orange')

plt.xlabel('rho')
plt.ylabel('T')
plt.title('Спинодаль и бинодаль Ван-дер-Ваальса на фоне измеренных границ')
plt.xlim(left=0.0, right=max(0.9, float(rho_grid.max()) * 1.02))
plt.ylim(bottom=0.0, top=max(float(critical_T) * 1.08, float(eos_mean['T_target'].max()) * 1.03))
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
path = FIGURES_DIR / 'vdw_spinodal_vs_measured_boundaries.png'
plt.savefig(path, dpi=160)
plt.show()
print('vdW a:', a_fit)
print('vdW b:', b_fit)
print('vdW critical rho:', critical_rho)
print('vdW critical T:', critical_T)
print('vdW binodal rows:', len(vdw_binodal_df))
print('saved:', path)
print('saved:', PROCESSED_DIR / 'vdw_spinodal_curve.csv')
print('saved:', PROCESSED_DIR / 'vdw_binodal_curve.csv')


In [ ]:
# Визуализация энергии: U/N по плотности для каждой температуры.
plt.figure(figsize=(8, 5))
for T, part in eos_mean.groupby('T_target'):
    part = part.sort_values('rho_target')
    plt.plot(part['rho_target'], part['U_mean'], marker='o', linewidth=1.4, label='T=' + format(float(T), '.2f'))
plt.xlabel('rho')
plt.ylabel('U/N')
plt.title('Potential energy per particle')
plt.grid(alpha=0.3)
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
path = FIGURES_DIR / 'potential_energy_isotherms.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)


In [ ]:
# Простая карта покрытия: какие T/rho уже есть в данных.
coverage_grid = eos_ok.pivot_table(index='T_target', columns='rho_target', values='run_id', aggfunc='count', fill_value=0)
plt.figure(figsize=(10, 4))
plt.imshow(coverage_grid.values, aspect='auto', interpolation='nearest')
plt.colorbar(label='number of seeds')
plt.xticks(range(len(coverage_grid.columns)), [format(float(x), '.4g') for x in coverage_grid.columns], rotation=45, ha='right')
plt.yticks(range(len(coverage_grid.index)), [format(float(x), '.2f') for x in coverage_grid.index])
plt.xlabel('rho')
plt.ylabel('T')
plt.title('EOS coverage')
plt.tight_layout()
path = FIGURES_DIR / 'coverage_heatmap.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)


In [ ]:
print('Processed files:')
for path in sorted(PROCESSED_DIR.glob('*.csv')):
    print(path, path.stat().st_size, 'bytes')
print('Figure files:')
for path in sorted(FIGURES_DIR.glob('*.png')):
    print(path, path.stat().st_size, 'bytes')
